# 02 Preprocessing

This notebook performs an initial data quality review on the raw dataset before any cleaning steps are applied.

Checks covered:
- missing values
- inferred data types
- category validation
- suspicious or out-of-place values

In [1]:
from pathlib import Path
import csv

dataset_path = Path("../data/raw/ElectraHub_Data.csv")
with dataset_path.open(newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    columns = reader.fieldnames

print(f"Dataset path: {dataset_path}")
print(f"Rows loaded: {len(rows)}")
print(f"Columns loaded: {len(columns)}")

Dataset path: ../data/raw/ElectraHub_Data.csv\n
Rows loaded: 3000\n
Columns loaded: 16\n

In [2]:
missing_counts = {column: 0 for column in columns}
for row in rows:
    for column in columns:
        value = row[column]
        if value is None or value.strip() == "":
            missing_counts[column] += 1

print("Missing values by column:")
for column, count in missing_counts.items():
    print(f"- {column}: {count}")

Missing values by column:\n
- Region: 0\n
- Product_Category: 0\n
- Campaign_Type: 0\n
- Product_Age_Months: 0\n
- Product_Price: 0\n
- Competitor_Price_Index: 0\n
- Advertising_Expenditure: 0\n
- Discount_Percentage: 0\n
- Campaign_Engagement_Score: 0\n
- Inventory_Level: 0\n
- Num_Reviews: 0\n
- Avg_Customer_Rating: 0\n
- Return_Rate: 0\n
- Length_Product_Description: 0\n
- Popularity: 0\n
- Sales: 0\n

## Missing-Value Finding

There are no missing values in any of the 16 columns. That means we do not need imputation for this version of the dataset.

In [3]:
def is_numeric_column(column_name):
    for row in rows:
        value = row[column_name].strip()
        if value == "":
            continue
        try:
            float(value)
        except ValueError:
            return False
    return True

integer_like_columns = {
    "Product_Age_Months",
    "Inventory_Level",
    "Num_Reviews",
    "Length_Product_Description",
}

print("Inferred data type review:")
for column in columns:
    if is_numeric_column(column):
        if column in integer_like_columns:
            label = "numeric (integer-like)"
        else:
            label = "numeric (float)"
            if column == "Sales":
                label = "numeric (float target)"
    else:
        label = "categorical/string"
        if column == "Popularity":
            label = "categorical/string (ordinal)"
    print(f"- {column}: {label}")

Inferred data type review:\n
- Region: categorical/string\n
- Product_Category: categorical/string\n
- Campaign_Type: categorical/string\n
- Product_Age_Months: numeric (integer-like)\n
- Product_Price: numeric (float)\n
- Competitor_Price_Index: numeric (float)\n
- Advertising_Expenditure: numeric (float)\n
- Discount_Percentage: numeric (float)\n
- Campaign_Engagement_Score: numeric (float)\n
- Inventory_Level: numeric (integer-like)\n
- Num_Reviews: numeric (integer-like)\n
- Avg_Customer_Rating: numeric (float)\n
- Return_Rate: numeric (float)\n
- Length_Product_Description: numeric (integer-like)\n
- Popularity: categorical/string (ordinal)\n
- Sales: numeric (float target)\n

## Data Type Finding

The observed values match the expected types well. There are no obvious type mismatches such as text inside numeric fields or numeric codes where labels were expected.

Columns that should be treated carefully during preprocessing:
- `Popularity` is categorical, but it is also ordinal.
- `Product_Age_Months`, `Inventory_Level`, `Num_Reviews`, and `Length_Product_Description` are stored as numeric values and behave like counts.
- `Sales` should be treated as the target variable, not as an input feature.

In [4]:
categorical_columns = ["Region", "Product_Category", "Campaign_Type", "Popularity"]

print("Category validation:")
for column in categorical_columns:
    categories = sorted({row[column] for row in rows})
    print(f"{column}: {categories}")

duplicate_rows = len(rows) - len({tuple(row[column] for column in columns) for row in rows})
print(f"\nDuplicate rows: {duplicate_rows}")

Category validation:\n
Region: ['East', 'North', 'South', 'West']\n
Product_Category: ['Mobile', 'Tablet']\n
Campaign_Type: ['Email', 'Influencer', 'Search Ad', 'Social Media']\n
Popularity: ['High', 'Low', 'Moderate', 'Very High', 'Very Low']\n
\n
Duplicate rows: 0\n

In [5]:
rule_checks = {
    "Product_Age_Months <= 0": sum(float(row["Product_Age_Months"]) <= 0 for row in rows),
    "Product_Price <= 0": sum(float(row["Product_Price"]) <= 0 for row in rows),
    "Competitor_Price_Index <= 0": sum(float(row["Competitor_Price_Index"]) <= 0 for row in rows),
    "Advertising_Expenditure < 0": sum(float(row["Advertising_Expenditure"]) < 0 for row in rows),
    "Discount_Percentage outside 0-100": sum(not (0 <= float(row["Discount_Percentage"]) <= 100) for row in rows),
    "Campaign_Engagement_Score outside 0-100": sum(not (0 <= float(row["Campaign_Engagement_Score"]) <= 100) for row in rows),
    "Inventory_Level < 0": sum(float(row["Inventory_Level"]) < 0 for row in rows),
    "Num_Reviews < 0": sum(float(row["Num_Reviews"]) < 0 for row in rows),
    "Avg_Customer_Rating outside 0-5": sum(not (0 <= float(row["Avg_Customer_Rating"]) <= 5) for row in rows),
    "Return_Rate < 0": sum(float(row["Return_Rate"]) < 0 for row in rows),
    "Length_Product_Description <= 0": sum(float(row["Length_Product_Description"]) <= 0 for row in rows),
    "Sales < 0": sum(float(row["Sales"]) < 0 for row in rows),
}

print("Rule-based suspicious value checks:")
for check_name, count in rule_checks.items():
    print(f"- {check_name}: {count}")

print("\nInteger-like column check:")
for column in sorted(integer_like_columns):
    non_integer_count = sum(not float(row[column]).is_integer() for row in rows)
    print(f"- {column} non-integer values: {non_integer_count}")

Rule-based suspicious value checks:\n
- Product_Age_Months <= 0: 0\n
- Product_Price <= 0: 0\n
- Competitor_Price_Index <= 0: 0\n
- Advertising_Expenditure < 0: 0\n
- Discount_Percentage outside 0-100: 0\n
- Campaign_Engagement_Score outside 0-100: 0\n
- Inventory_Level < 0: 0\n
- Num_Reviews < 0: 0\n
- Avg_Customer_Rating outside 0-5: 0\n
- Return_Rate < 0: 0\n
- Length_Product_Description <= 0: 0\n
- Sales < 0: 0\n
\n
Integer-like column check:\n
- Product_Age_Months non-integer values: 0\n
- Inventory_Level non-integer values: 0\n
- Num_Reviews non-integer values: 0\n
- Length_Product_Description non-integer values: 0\n

In [6]:
def iqr_outlier_count(column_name):
    values = sorted(float(row[column_name]) for row in rows)
    n = len(values)
    q1 = values[n // 4]
    q3 = values[(3 * n) // 4]
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return sum(value < lower_bound or value > upper_bound for value in values)

low_prices = sorted(float(row["Product_Price"]) for row in rows if float(row["Product_Price"]) < 50)

print("Potential edge cases to review manually:")
print(f"- Product_Price under $50: {len(low_prices)}")
print(f"- Product_Price examples (lowest 5): {low_prices[:5]}")
print(f"- Competitor_Price_Index outside IQR bounds: {iqr_outlier_count('Competitor_Price_Index')}")
print(f"- Campaign_Engagement_Score outside IQR bounds: {iqr_outlier_count('Campaign_Engagement_Score')}")
print(f"- Num_Reviews outside IQR bounds: {iqr_outlier_count('Num_Reviews')}")
print(f"- Avg_Customer_Rating outside IQR bounds: {iqr_outlier_count('Avg_Customer_Rating')}")
print(f"- Return_Rate outside IQR bounds: {iqr_outlier_count('Return_Rate')}")
print("\nThese are statistical outliers, not confirmed errors.")

Potential edge cases to review manually:\n
- Product_Price under $50: 55\n
- Product_Price examples (lowest 5): [10.11, 14.96, 15.66, 15.89, 15.92]\n
- Competitor_Price_Index outside IQR bounds: 9\n
- Campaign_Engagement_Score outside IQR bounds: 15\n
- Num_Reviews outside IQR bounds: 119\n
- Avg_Customer_Rating outside IQR bounds: 17\n
- Return_Rate outside IQR bounds: 22\n
\n
These are statistical outliers, not confirmed errors.\n

## Preprocessing Findings

- No missing values were found, so imputation is not required.
- The inferred data types look appropriate for every column.
- No invalid category labels were found in `Region`, `Product_Category`, `Campaign_Type`, or `Popularity`.
- No exact duplicate rows were found.
- No impossible numeric values were found based on simple business rules such as negative prices, ratings above 5, or percentages outside expected ranges.
- A small number of values are worth reviewing as edge cases rather than outright errors. The clearest example is `55` products priced under `$50`, which may be plausible in a synthetic dataset but would be unusual for mobile and tablet products in many real retail settings.
- Several columns contain statistical outliers by IQR, especially `Num_Reviews`, but these do not automatically indicate bad data. They may represent naturally skewed business behavior.
- Overall, the dataset appears very clean and possibly synthetic, so the next preprocessing steps will likely focus more on encoding and scaling than on heavy data repair.

## Cleaning Applied

The cleaning step below keeps the dataset conservative and does not apply encoding, scaling, or outlier treatment.

What it does:
- trims whitespace from text fields
- casts each column to its intended type
- removes exact duplicate rows if any appear
- saves the cleaned dataset to `../data/processed/ElectraHub_Data_cleaned.csv`

What it does not do:
- no missing-value imputation
- no category encoding
- no feature scaling
- no outlier removal

In [7]:
from pathlib import Path
import csv

input_path = Path("../data/raw/ElectraHub_Data.csv")
output_path = Path("../data/processed/ElectraHub_Data_cleaned.csv")

int_columns = {
    "Product_Age_Months",
    "Inventory_Level",
    "Num_Reviews",
    "Length_Product_Description",
}
float_columns = {
    "Product_Price",
    "Competitor_Price_Index",
    "Advertising_Expenditure",
    "Discount_Percentage",
    "Campaign_Engagement_Score",
    "Avg_Customer_Rating",
    "Return_Rate",
    "Sales",
}
string_columns = {
    "Region",
    "Product_Category",
    "Campaign_Type",
    "Popularity",
}

with input_path.open(newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    raw_rows = list(reader)

cleaned_rows = []
seen_rows = set()
for row in raw_rows:
    cleaned_row = {}
    for column in columns:
        value = row[column].strip()
        if column in int_columns:
            cleaned_row[column] = int(float(value))
        elif column in float_columns:
            cleaned_row[column] = float(value)
        elif column in string_columns:
            cleaned_row[column] = value
        else:
            cleaned_row[column] = value

    row_key = tuple(cleaned_row[column] for column in columns)
    if row_key not in seen_rows:
        seen_rows.add(row_key)
        cleaned_rows.append(cleaned_row)

with output_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for row in cleaned_rows:
        writer.writerow(row)

column_types = {}
for column in columns:
    if column in int_columns:
        column_types[column] = "int"
    elif column in float_columns:
        column_types[column] = "float"
    else:
        column_types[column] = "string"

print(f"Cleaned dataset saved to: {output_path}")
print(f"Cleaned shape: {len(cleaned_rows)} rows x {len(columns)} columns")
print("Column types after cleaning:")
for column in columns:
    print(f"- {column}: {column_types[column]}")

Cleaned dataset saved to: ../data/processed/ElectraHub_Data_cleaned.csv\n
Cleaned shape: 3000 rows x 16 columns\n
Column types after cleaning:\n
- Region: string\n
- Product_Category: string\n
- Campaign_Type: string\n
- Product_Age_Months: int\n
- Product_Price: float\n
- Competitor_Price_Index: float\n
- Advertising_Expenditure: float\n
- Discount_Percentage: float\n
- Campaign_Engagement_Score: float\n
- Inventory_Level: int\n
- Num_Reviews: int\n
- Avg_Customer_Rating: float\n
- Return_Rate: float\n
- Length_Product_Description: int\n
- Popularity: string\n
- Sales: float\n